# 频散反演结果与正演模型对比重绘

读取已有反演CSV，比较正演真值、反演起始模型及恢复结果。这里不重跑Qt反演。


当前显示：道集gray、模型turbo、波场coolwarm。15/30 Hz默认9时刻全域Vz，统一±20 µm/s对称对数色标；原始DEM保留jet，频散谱独立使用viridis。参数见同目录styles/current_publication.json。


## 输入与输出约定

这里的代码只读输入，不重跑模拟，不修改原始SEG-Y/VTS。路径可以是绝对路径；相对路径以项目外层工作区为基准。生成的图件、CSV、缓存及执行副本统一放入外层`out`的子目录。

如果输入不是默认文件夹结构，可在配置中为每个算例单独填写`record_files`、`geometry_csv`、`model_vts`、`snapshot_index_csv`和`snapshot_dir`，详见同目录README。源码中的Notebook不保存运行输出；需要保留执行结果时请另存到`out`。


In [ ]:
import sys, json
from pathlib import Path
sys.dont_write_bytecode = True

# 如果Notebook从其他位置启动，可在这里指定仓库根目录。
REPOSITORY_ROOT = None
if REPOSITORY_ROOT is None:
    candidates = [p for q in [Path.cwd(), *Path.cwd().parents]
                  for p in [q, q / "SeisWaveLab_Curvilinear"]
                  if (p / "tools/seismic_plotting/seisplot/core.py").is_file()]
    if not candidates:
        raise FileNotFoundError("请将REPOSITORY_ROOT设为Git仓库根目录")
    REPOSITORY_ROOT = candidates[0].resolve()
else:
    REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
WORKSPACE = REPOSITORY_ROOT.parent
TOOL_DIR = REPOSITORY_ROOT / "tools/seismic_plotting"
sys.path.insert(0, str(TOOL_DIR))
from seisplot.core import load_config, gallery
SHOW_INLINE = True


In [ ]:
from seisplot.inversion import InversionStudy
CONFIG = load_config(TOOL_DIR / "examples/inversion.json")
INVERSION_INPUT_DIR = WORKSPACE / "out/surface_wave_masw/analysis_20261024/inversion"
OUTPUT_DIR = WORKSPACE / "out/replots/inversion"
CONFIG["input_dir"] = str(INVERSION_INPUT_DIR)
CONFIG["output_dir"] = str(OUTPUT_DIR)
CONFIG["results_csv"] = ["results.csv", "refined_results.csv"]
CONFIG["main_id"] = "main_fixed_start2"
CONFIG["fit_csv"] = "results.csv_main_fixed_start2_fit.csv"
# 使用新的模型时，同时修改真值、起始值和哪些层参与求解。
CONFIG["truth"] = {"thickness_m": [10, 20, 40], "vs_m_s": [250, 500, 800, 1250]}
CONFIG["initial_vs_m_s"] = [280, 480, 800, 1250]
CONFIG["free_vs"] = [True, True, False, False]

# 显示样式由styles/current_publication.json统一维护；修改CONFIG可显式覆盖。
print("显示配置:", CONFIG["style_profile"], CONFIG["style_provenance"]["version"])


## 1. 模型恢复与逐层误差

固定先验以灰底标明，误差表不将固定层的零偏差计为恢复精度。更换分层数时，各参数数组长度应一致。


In [ ]:
study = InversionStudy(CONFIG)
study.plot_models()


## 2. 观测曲线、初始与反演正演曲线

读取已有拟合曲线，检查其RMSE是否与选定模型行一致。


In [ ]:
study.plot_curve_fit()


## 3. 多初值与非唯一性

配置multistart_groups中的结果ID即可改画新的初值试验。


In [ ]:
study.plot_multistart()


## 4. 厚度扫描

绘制已有厚度扫描结果。该功能不自动搜索层厚，也不代表所有深层界面已独立恢复。


In [ ]:
study.plot_thickness_scan()


## 5. 网格、频带与偏移距窗口影响


In [ ]:
study.plot_sensitivity()


In [ ]:
print("输出目录：", study.out)
if SHOW_INLINE:
    gallery(study.out, "inversion*.png", limit=3)
